Basically even though we have full rank contrabilit just tell us that a node being excited can cause the others to be excited. If our k val is high that means that all the nodes move together. But we have full rank since one node push causes other to move. But what if my idea is if we keep reducing k val and checking rank. Whatever the last full rank k val is prob useful since that means our push has full influence to nearby nodes but at the same time we don't have them move together.

Since if k is too high they move collectively but too low they are isolated from each other. But now we can find what's perfect k val.

In [15]:
import numpy as np
from scipy.linalg import solve_continuous_lyapunov


def compute_controllability_gramian(X_steady, epsilons, kappa, zeta, B_matrix):
    N = len(X_steady)

    # 1. Compute local stiffness derivative for each node
    k_local = np.zeros(N)
    for i in range(N):
        Xi = X_steady[i]
        eps_i = epsilons[i]
        k_local[i] = -2 * (1 - (1 + eps_i) / (1 + Xi**2) ** 1.5)

    # 2. Build the N x N effective stiffness matrix K_eff
    K_eff = np.zeros((N, N))
    for i in range(N):
        deg = 2 if (0 < i < N - 1) else 1
        K_eff[i, i] = k_local[i] - deg * kappa
        if i > 0:
            K_eff[i, i - 1] = kappa
        if i < N - 1:
            K_eff[i, i + 1] = kappa

    # 3. Build the full 2N x 2N system matrix A
    A = np.block([[np.zeros((N, N)), np.eye(N)], [K_eff, -2 * zeta * np.eye(N)]])

    # B matrix needs to match the full state size (2N x m)
    # Here B_matrix passed is already 2N x 1

    # 4. Solve the Continuous Lyapunov Equation for the Gramian Wc:
    # A * Wc + Wc * A^T + B * B^T = 0
    # (This is the analytical solution for the infinite-time integral Gramian)
    Q = B_matrix @ B_matrix.T
    Wc = solve_continuous_lyapunov(A, -Q)

    return Wc, A


# --- Parameters ---
N = 10
epsilons = np.full(N, 0.1)
zeta = 0.5
state = np.full(N, 0.4583)  # Stable "All Up" well

# Actuator pushing on Node 1 (velocity V1)
B_matrix = np.zeros((2 * N, 1))
B_matrix[N, 0] = 1.0

# --- Sweep Different Coupling Values (Kappa) ---
kappa_values = [2.0, 1.0, 0.5, 0.1, 0.01]

print(f"{'Coupling (κ)':<15} | {'Min Eigenvalue of Wc':<22} | {'Condition Number'}")
print("-" * 60)

for kappa in kappa_values:
    Wc, A = compute_controllability_gramian(state, epsilons, kappa, zeta, B_matrix)

    # Eigenvalues of the Gramian tell us how hard/easy it is to reach certain states
    eigenvals = np.linalg.eigvalsh(Wc)
    min_eigenval = np.min(eigenvals)  # Smallest eigenvalue (hardest direction to steer)

    # Condition number tells us how "skewed" the control effort is
    cond_num = np.linalg.cond(Wc)

    print(f"{kappa:<15.2f} | {min_eigenval:<22.4e} | {cond_num:.4e}")

Coupling (κ)    | Min Eigenvalue of Wc   | Condition Number
------------------------------------------------------------
2.00            | 7.7368e-10             | 5.7438e+08
1.00            | 2.6685e-11             | 2.1455e+10
0.50            | 8.7761e-14             | 8.4358e+12
0.10            | -1.6597e-17            | 3.2985e+18
0.01            | -1.0468e-16            | 1.1944e+19


In [16]:
import numpy as np

# Assuming N = 10, N_states = 20
# Let's build the controllability matrix C for a given kappa
N = 10
zeta = 0.5
epsilons = np.full(N, 0.1)
state = np.full(N, 0.4583)
kappa = 0.5  # Try changing this to 0.01 to see the explosion

k_local = -2 * (1 - (1 + epsilons) / (1 + state**2) ** 1.5)
K_eff = np.zeros((N, N))
for i in range(N):
    deg = 2 if (0 < i < N - 1) else 1
    K_eff[i, i] = k_local[i] - deg * kappa
    if i > 0:
        K_eff[i, i - 1] = kappa
    if i < N - 1:
        K_eff[i, i + 1] = kappa

A = np.block([[np.zeros((N, N)), np.eye(N)], [K_eff, -2 * zeta * np.eye(N)]])

B_matrix = np.zeros((2 * N, 1))
B_matrix[N, 0] = 1.0  # Actuator on V1

# Construct Kalman Controllability Matrix C
n_states = A.shape[0]
C = B_matrix
current_term = B_matrix
for i in range(1, n_states):
    current_term = A @ current_term
    C = np.hstack((C, current_term))

# Check condition number of C (relates directly to pseudo-inverse stability)
cond_C = np.linalg.cond(C)
print(f"Condition number of Controllability Matrix C at kappa={kappa}: {cond_C:.4e}")

# Compute pseudo-inverse to see magnitude of input reconstruction scaling
C_pinv = np.linalg.pinv(C)
print(f"Max value in pseudo-inverse matrix: {np.max(np.abs(C_pinv)):.4e}")

Condition number of Controllability Matrix C at kappa=0.5: 4.9968e+09
Max value in pseudo-inverse matrix: 6.0844e+06


In [21]:
import numpy as np

# --- 1. Setup System Parameters ---
N = 10
zeta = 0.5
kappa = .5  # Our sweet spot!
epsilons = np.full(N, 0.1)
state = np.full(N, 0.4583)  # Stable "All Up" well

# Build A matrix
k_local = -2 * (1 - (1 + epsilons) / (1 + state**2) ** 1.5)
K_eff = np.zeros((N, N))
for i in range(N):
    deg = 2 if (0 < i < N - 1) else 1
    K_eff[i, i] = k_local[i] - deg * kappa
    if i > 0:
        K_eff[i, i - 1] = kappa
    if i < N - 1:
        K_eff[i, i + 1] = kappa

A = np.block([[np.zeros((N, N)), np.eye(N)], [K_eff, -2 * zeta * np.eye(N)]])

# Define a "True" B matrix (Actuator pushing ONLY on Node 1 velocity, index N)
B_true = np.zeros((2 * N, 1))
B_true[N, 0] = 1.0

# --- 2. Generate Synthetic Experimental Data ---
# Imagine we run an experiment: we shake Node 1 randomly and record the states over time.
np.random.seed(42)
n_steps = 1000
dt = 0.005

X_data = np.zeros((2 * N, n_steps))
X_dot_data = np.zeros((2 * N, n_steps))
U_data = np.zeros((1, n_steps))  # 1 control input signal

X_current = np.zeros(2 * N)  # Start near equilibrium

for t in range(n_steps):
    # Random input force applied at step t
    u_t = np.random.randn(1) * 5.0
    U_data[:, t] = u_t

    # True physical dynamics: X_dot = A*X + B*u
    x_dot = A @ X_current + (B_true @ u_t)

    # Save snapshot
    X_data[:, t] = X_current
    X_dot_data[:, t] = x_dot

    # Step forward in time (Euler integration)
    X_current = X_current + x_dot * dt

# --- 3. Solve for B Using the Pseudo-Inverse ---
# Equation: B = (X_dot - A*X) * U^dagger
Y_residual = X_dot_data - (A @ X_data)

# Use Moore-Penrose pseudo-inverse to solve least-squares regression
B_estimated = Y_residual @ np.linalg.pinv(U_data)

# --- 4. Evaluate Results ---
estimation_error = np.linalg.norm(B_true - B_estimated)

print("--- System Identification Results ---")
print(f"Total Estimation Error (Frobenius Norm): {estimation_error:.4e}")
print(f"True B value at Actuator Slot (V1):      {B_true[N, 0]:.4f}")
print(f"Estimated B value from Data (V1):        {B_estimated[N, 0]:.4f}")

# Let's check if any "ghost" forces leaked into other nodes due to noise/approximation
max_ghost_force = np.max(np.abs(np.delete(B_estimated, N)))
print(f"Max leakage/ghost force on other nodes:  {max_ghost_force:.4e}")

--- System Identification Results ---
Total Estimation Error (Frobenius Norm): 6.6613e-16
True B value at Actuator Slot (V1):      1.0000
Estimated B value from Data (V1):        1.0000
Max leakage/ghost force on other nodes:  2.6452e-21
